# Exploring data using pandas

Our first task in this week's lesson is to learn how to **read and explore data files in Python**. We will focus on using [pandas](https://pandas.pydata.org/pandas-docs/stable/) which is an open-source package for data analysis in Python. pandas is an excellent toolkit for working with **real world data** that often have a tabular structure (rows and columns).

We will first get familiar with the **pandas data structures**: *DataFrame* and *Series*:


- **pandas DataFrame** (a 2-dimensional data structure) is used for storing and manipulating table-like data (data with rows and columns) in Python. You can think of a pandas DataFrame as a programmable spreadsheet. 
- **pandas Series** (a 1-dimensional data structure) is used for storing and manipulating a sequence of values. pandas Series is kind of like a list, but more clever. One row or one column in a pandas DataFrame is actually a pandas Series. 

These pandas structures incorporate a number of things we've already encountered, such as indices, data stored in a collection, and data types.

Both DataFrames and Series in pandas have an index that can be used to select values, but they also have column labels to identify columns in DataFrames. In the lesson this week we'll use many of these features to explore real-world data and learn some useful data analysis procedures.

For a comprehensive overview of pandas data structures you can have a look at Chapter 5 in Wes McKinney's book [Python for Data Analysis (2nd Edition, 2017)](https://geo-python-site.readthedocs.io/en/latest/course-info/resources.html#books) and the [pandas online documentation about data structures](https://pandas.pydata.org/pandas-docs/stable/user_guide/dsintro.html). 

> **Note:**
>
> pandas is a "high-level" package, which means that it makes use of several other packages, such as [NumPy](https://numpy.org/), in the background. There are several ways in which data can be read from a file in Python, and for several years now we have decided to focus primarily on pandas because it is easy-to-use, efficient and intuitive. If you are curious about other approaches for interacting with data files, you can find lesson materials from previous years about reading data using [NumPy](https://geo-python-site.readthedocs.io/en/2018.1/notebooks/L5/numpy/1-Exploring-data-using-numpy.html#Reading-a-data-file-with-NumPy) or [built-in Python functions](https://geo-python-site.readthedocs.io/en/2017.1/lessons/L5/reading-data-from-file.html).


- File name: `3172914.txt` (have a look at the file before reading it in using pandas!). It contains daily temperatures for March 2022 from a Swedish weather station in NOAA's GHCN-Daily dataset. The temperatures (`TAVG`, `TMAX`, `TMIN`) are in **degrees Fahrenheit**.

- This lecture has adapted or reused material from University of Helsinki Geo-python course (https://geo-python-site.readthedocs.io/en/latest/course-info/licensing.html ) under a Creative Commons Attribution-ShareAlike 4.0 International licence. 



## Reading a data file with pandas

Now we're ready to read in our temperature data file. **First, we need to import the pandas module.** It is customary to import pandas as `pd`:

In [ ]:
# Import pandas. It is customary to give it the short nickname 'pd'.
import pandas as pd

**Next, we'll read the input data file** and store the contents of that file in a variable called `df`, using the `pandas.read_csv()` function:

In [ ]:
# Read the text file with pandas (first attempt)
df = pd.read_csv('3172914.txt')

In [ ]:
# Everything has ended up squeezed into a single column - not what we want
print(df)

> ⚠️ **Common Pitfall: `FileNotFoundError` and the wrong delimiter**
>
> Two things go wrong for almost everyone when reading a data file:
>
> - **The file must be where Python is looking.** `pd.read_csv('3172914.txt')` only works if that file sits in the *same folder as this notebook*. Otherwise you get a `FileNotFoundError` - either give the full path, or move the file next to the notebook.
> - **Check the separator.** `read_csv` assumes **commas** by default. If your file uses spaces, tabs, or semicolons, everything lands in a single column (exactly as it did above). Always open the file in a text editor first and look at how the values are separated.

While it could look OK, everything appears to be in one column. We don't want that. Instead we need separate columns for each station, date, average, max and min temperatures. We can fix that by specifying the correct delimiter. It seems that pandas by default does not recognize empty space as a delimiter. Let's try again!

In [ ]:
# Try again, telling pandas that the columns are separated by a space
df = pd.read_csv('3172914.txt', delimiter=' ')

In [ ]:
# Better, but now pandas thinks there are far too many columns
print(df)

Something is wrong again. We definitely do not have 33 columns, but only 5. (Every single space now counts as a separator, so each run of spaces creates extra, empty columns.) Maybe it is better to specify them ourselves, since we know their names. Let's make a list and add some more parameters to the pandas import!

In [ ]:
# Give the columns names ourselves, and say which ones we want to keep
columns = ['STATION', 'DATE', 'TAVG', 'TMAX', 'TMIN']
usecols = [0, 1, 2, 3, 4]

In [ ]:
# Read again, this time supplying our own column names
df = pd.read_csv('3172914.txt', delimiter=' ', names=columns, usecols=usecols)

In [ ]:
# Closer, but TMAX and TMIN are still coming out as NaN (missing)
print(df)

The issue is that the columns are separated by a *varying* number of spaces. With `delimiter=' '`, every extra space creates an extra (empty) column, so the `TMAX` and `TMIN` values end up in columns that we did not select, and the columns we did select are filled with `NaN`. The `skipinitialspace=True` parameter tells pandas to ignore the extra spaces that follow a delimiter.

In [ ]:
# skipinitialspace=True tells pandas to ignore the extra spaces after each delimiter
df = pd.read_csv('3172914.txt', delimiter=' ', names=columns, usecols=usecols, skipinitialspace=True)

In [ ]:
# The temperature columns are now read correctly
print(df)

Much better now! But the header line of the file now appears as the first row of data. This happens because we gave our own column names with `names=...`, so pandas treats the first line of the file as data. We can skip that line with `skiprows=1`.

In [ ]:
# Read the file one more time, now also skipping the first row (skiprows=1),
# which contains the file's own header line (we supply the column names ourselves).
df = pd.read_csv('3172914.txt', delimiter=' ', skiprows=1, names=columns, usecols=usecols, skipinitialspace=True)

In [ ]:
# Now the header line is no longer read as data, and the data looks right
print(df)

If all goes as planned, you should now have a new variable `df` in memory that contains the input data. 

Let's check the contents of this variable by calling `df` or `print(df)`:

In [ ]:
# Display the DataFrame
df

This looks OK, but there are some strange values present such as `NaN`.

`NaN` stands for "not a number", and might indicate some problem with reading in the contents of the file or it can affect our analysis if we are unaware of them.

> ⚠️ **Common Pitfall: `NaN` is not zero**
>
> `NaN` means "not a number" - a *missing* value. It is tempting to assume missing values are harmless, but they are not:
>
> - `NaN` often means the file was read incorrectly (wrong delimiter or unexpected spacing), so check your `read_csv` settings first.
> - Missing values quietly change your statistics. By default pandas **skips** `NaN` when computing things like `mean()`, so an average can be based on far fewer rows than you think.
>
> Use `df.dropna()` to remove rows with missing values - but remember it returns a **new** DataFrame, so you must store the result: `df = df.dropna()`.

> **Tip: a shorter solution.** All of the steps above can be replaced by telling pandas that the columns are separated by *one or more* whitespace characters, written as the pattern `r'\s+'`. This reads the header, the five columns and the missing values correctly in one go - we will use this approach in Lecture 6.

In [ ]:
# sep=r'\s+' means 'one or more spaces (or tabs)' between the columns
df_short = pd.read_csv('3172914.txt', sep=r'\s+')
df_short.head()

After reading in the data, it is always good to check that everything went well by printing out the data as we did here. However, often it is enough to have a look at the top few rows of the data. 

We can use the `.head()` method of the pandas DataFrame object to quickly check the top rows. By default, the `.head()` function returns the first 5 rows of the DataFrame:

In [ ]:
# .head() shows the first 5 rows by default
df.head()

We can also check the last rows of the data using `df.tail()`:

In [ ]:
# .tail() shows the last 5 rows
df.tail()

Note that pandas DataFrames have **labelled axes (rows and columns)**. In our sample data, the rows labeled with an index value (`0` to `30`), and columns labelled `STATION`, `DATE`,`TAVG`, `TMAX`, and `TMIN`. Later on, we will learn how to use these labels for selecting and updating subsets of the data.

**Let's also confirm the data type of our data variable:**

In [ ]:
# Confirm the type: this is a pandas DataFrame
type(df)

No surprises here, our data variable is a pandas DataFrame.

#### Check your understanding

Read the file `3172914.txt` in again and store its contents in a new variable called `temp_data`. In this case you should only read in the columns `DATE` and `TAVG`, so the new variable `temp_data` should have 31 rows and 2 columns. You can achieve this using the `usecols` parameter when reading in the file. Feel free to check for more help in the [pandas.read_csv documentation](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html).

In [ ]:
# Solution
# Read only the DATE and TAVG columns into a NEW variable called temp_data,
# so that our full data set in df is left untouched.
columns = ['DATE','TAVG']
usecols = [1, 2]

temp_data = pd.read_csv('3172914.txt', delimiter=' ', skiprows=1, names=columns, usecols=usecols, skipinitialspace=True)

In [ ]:
# temp_data should have 31 rows and 2 columns
print(temp_data)

## DataFrame properties

Let's continue with the full data set that we have stored in the variable `df` and explore its contents further. 
A normal first step when you load new data is to explore the dataset a bit to understand how the data is structured, and what kind of values are stored in there.

Let's start by checking the size of our data frame. We can use the `len()` function similar to the one we use with lists to check how many rows we have:

In [ ]:
# Read the full data set into df again, so we have all five columns to explore
columns = ['STATION', 'DATE', 'TAVG', 'TMAX', 'TMIN']
usecols = [0, 1, 2, 3, 4]
df = pd.read_csv('3172914.txt', delimiter=' ', skiprows=1, names=columns, usecols=usecols, skipinitialspace=True)

In [ ]:
# len() gives the number of rows in the DataFrame
len(df)

Let's remove the NaN values. Using the `dropna()` method we can remove the rows that contain missing data.


In [ ]:
# dropna() returns a NEW DataFrame with the rows containing NaN removed,
# so we store the result back into df.
df = df.dropna()
print(df)

In [ ]:
# Dropping rows leaves gaps in the index, so renumber the rows from 0 again
df = df.reset_index(drop=True)

In [ ]:
# The index now runs 0, 1, 2, ... without gaps
print(df)

In [ ]:
# Check the number of rows
len(df)

In [ ]:
# Print the cleaned data one more time
print(df)

We can also get a quick sense of the size of the dataset using the `shape` attribute.


In [ ]:
# Check dataframe shape (number of rows, number of columns)
df.shape

Here we see that our dataset has 18 rows and 5 columns, just as we saw above when printing out the entire DataFrame.

> **Note:**
>
> `shape` is one of the several [attributes related to a pandas DataFrame](https://pandas.pydata.org/pandas-docs/stable/reference/frame.html#attributes-and-underlying-data).

**We can also check the column names we have in our DataFrame.** We already saw the column names when we checked the 5 first rows using `df.head()`, but often it is useful to access the column names directly. You can check the column names by calling `df.columns` (returns an index object that contains the column labels) or `df.columns.values`:

In [ ]:
# Print column names
df.columns.values

We can also find information about the row identifiers using the `index` attribute:

In [ ]:
# Print index
df.index

Here we see how the data is indexed: starting at 0 and stopping *before* 18 (so the last row has the index 17), with an increment of 1 between each value. This is basically the same way in which Python lists are indexed, however, pandas also allows other ways of identifying the rows. DataFrame indices could, for example, be character strings, or date objects. We will learn more about resetting the index later.

What about the data types of each column in our DataFrame? We can check the data type of all columns at once using `pandas.DataFrame.dtypes`:

In [ ]:
# Print data types
df.dtypes

Here we see that `STATION` contains text (shown as `object` or `str`, depending on your pandas version), `DATE` and `TAVG` are integers (`int64`), and `TMAX` and `TMIN` are decimal numbers (`float64`). `TMAX` and `TMIN` became decimal numbers because these columns contained missing values (`NaN`), which pandas stores as decimals.

#### Check your understanding

See if you can find a way to print out the number of columns in our DataFrame.

In [ ]:
# Here is one solution
len(df.columns)

## Selecting columns

We can select specific columns based on the column values. The basic syntax is `dataframe[value]`, where value can be a single column name, or a list of column names. Let's start by selecting two columns, `'DATE'` and `'TAVG'`:

In [ ]:
# Select two columns by passing a LIST of column names
selection = df[["DATE", "TAVG"]]

In [ ]:
# The result is a new DataFrame containing just those two columns
selection

Let's also check the data type of this selection:

In [ ]:
# Check the type of the selection
type(selection)

The subset is still a pandas DataFrame, and we are able to use all the methods and attributes related to a pandas DataFrame also with this subset. For example, we can check the shape:

In [ ]:
# Shape of the selection: (number of rows, number of columns)
selection.shape

We can also access a single column of the data based on the column name:

In [ ]:
# Selecting a SINGLE column with one pair of brackets gives a Series
df["TAVG"]

What about the type of the column itself?

In [ ]:
# Check datatype of the column
type(df["TAVG"])

**Each column (and each row) in a pandas data frame is actually a pandas Series** - a 1 dimensional data structure!

> ⚠️ **Common Pitfall: one pair of brackets vs two**
>
> This small difference changes what you get back:
>
> - `df["TAVG"]` - **one** pair of brackets gives a **Series** (a single column, 1-dimensional).
> - `df[["TAVG"]]` - **two** pairs give a **DataFrame** (a table that happens to have one column).
>
> Both look similar when printed, but they behave differently. If a method complains that a Series has no such attribute (or the other way round), check how many brackets you used.

> **Note:**
>
> You can also retrieve a column using a different syntax:
>
> ``` 
> df.TAVG
> ```
>
> This syntax works only if the column name is a valid name for a Python variable (e.g. the column name should not contain whitespace).
> The syntax `df["column"]` works for all kinds of column names, so we recommend using this approach.

## Descriptive statistics

pandas DataFrames and Series contain useful methods for getting summary statistics. Available methods include `mean()`, `median()`, `min()`, `max()`, and `std()` (the standard deviation).

We could, for example, check the mean temperature in our input data. We check the mean for a single column (*Series*): 

In [ ]:
# Check mean value of a column
df["TAVG"].mean()

and for several columns at once (a *DataFrame*). Note that `df.mean()` on the whole DataFrame would fail, because the `STATION` column contains text:

In [ ]:
# Check mean value for two columns
df[["TAVG","TMAX"]].mean()

For an overview of the basic statistics for all attributes in the data, we can use the `describe()` method:


In [ ]:
# Get descriptive statistics
df.describe()

#### Check your understanding

It doesn't make much sense to print out descriptive statistics for the `DATE` column now that the values are stored as integer values (and not datetime objects, which we will learn about in later lessons). 

See if you can print out the descriptive statistics again, this time only for the columns `TAVG`, `TMAX`, and `TMIN`:

In [ ]:
# Here is the solution
df[["TAVG", "TMAX", "TMIN"]].describe()

## Extra: Very basic plots

Visualizing the data is a key part of data exploration, and pandas comes with a handful of plotting methods, which all rely on the [Matplotlib](https://matplotlib.org/) plotting library. 

For very basic plots, we don’t need to import Matplotlib separately. We can already create very simple plots using the `DataFrame.plot` method. 

Let's plot all the columns that contain values related to temperatures:

In [ ]:
# Plot the three temperature columns against the row index
df[["TAVG", "TMAX", "TMIN"]].plot()

## Extra: From lists to pandas objects

Most often we create pandas objects by reading in data from an external source, such as a text file. Here, we will briefly see how you can create pandas objects from Python lists. If you have long lists of numbers, for instance, creating a pandas Series will allow you to interact with these values more efficiently in terms of computing time.

In [ ]:
# Create pandas Series from a list
number_series = pd.Series([4, 5, 6, 7.0])
print(number_series)

Note that pandas is smart about the conversion, detecting a single floating point value (`7.0`) and assigning all values in the Series the data type `float64`.

If needed, you can also set a custom index when creating the object:

In [ ]:
# You can also supply your own index labels instead of 0, 1, 2, 3
number_series = pd.Series([4, 5, 6, 7.0], index=["a", "b", "c", "d"])
print(number_series)

In [ ]:
# Confirm the type: this is a pandas Series
type(number_series)

How about combining several lists as a DataFrame? Let's take a few Swedish cities and their coordinates and see how we could combine those lists into a pandas DataFrame:

In [ ]:
# Station names
stations = [
    "Karlstad",
    "Stockholm",
    "Malmo",
    "Gothenburg",
]

# Latitude coordinates of the weather stations (degrees north)
lats = [59.38, 59.33, 55.60, 57.71]

# Longitude coordinates of the weather stations (degrees east)
lons = [13.50, 18.07, 13.00, 11.97]

Often we indeed create pandas DataFrames by reading in data (e.g. using `pd.read_csv(filename.csv)`), but sometimes you might also combine lists into a DataFrame inside the script using the `pandas.DataFrame` constructor. Here, we are using a *Python dictionary* `{"column_1": list_1, "column_2": list_2, ...}` to indicate the structure of our data. 

In [ ]:
# Build a DataFrame from the three lists.
# The dictionary keys become the column names.
new_data = pd.DataFrame(data={"station_name": stations, "lat": lats, "lon": lons})
new_data

In [ ]:
# Confirm the type: this is a pandas DataFrame
type(new_data)

Often, you might start working with an empty data frame instead of existing lists:

In [ ]:
# Create a completely empty DataFrame.
# Note: this REPLACES the temperature data we had stored in df!
df = pd.DataFrame()

In [ ]:
# An empty DataFrame prints as "Empty DataFrame"
print(df)

Check more details about the available parameters and methods in [the pandas.DataFrame documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html#pandas-dataframe).

## Summary

This lesson introduced **pandas**, the main Python toolkit for working with tabular data. You can now:

- **Import pandas** with `import pandas as pd`.
- **Read a data file** with `pd.read_csv()`, adjusting `delimiter`, `names`, `usecols`, `skiprows`, and `skipinitialspace` until the columns come out correctly.
- **Inspect a DataFrame** with `.head()`, `.tail()`, `len()`, `.shape`, `.columns`, `.index`, and `.dtypes`.
- **Clean missing data** with `.dropna()`, and tidy the row numbering with `.reset_index(drop=True)`.
- **Select columns** with `df["column"]` (a Series) or `df[["col1", "col2"]]` (a DataFrame).
- **Compute statistics** with `.mean()`, `.median()`, `.min()`, `.max()`, `.std()`, and get an overview with `.describe()`.
- **Make a quick plot** with `.plot()`, and **build pandas objects from lists** with `pd.Series()` and `pd.DataFrame()`.

| Watch out for | Reminder |
| --- | --- |
| `FileNotFoundError` | The data file must be in the same folder as the notebook (or give the full path). |
| Everything in one column | The `delimiter` doesn't match how the file separates its values. |
| `NaN` values | Missing data - *not* zero. They are skipped in statistics, so check them before analysing. |
| Series vs DataFrame | `df["col"]` gives a Series; `df[["col"]]` gives a DataFrame. |

pandas is the workhorse of data analysis in Python - you will use these same few commands constantly from here on.